# Job Application Extractor Experiment
This notebook allows you to test the `EmailExtractor` with individual email data to see how the current model performs.

In [1]:
import sys
import os
from pathlib import Path
import json

# Check for missing dependencies and suggest installation
try:
    import sqlmodel
    import streamlit
    import pydantic
except ImportError:
    print("⚠️ Missing dependencies detected.")
    print("To install them in this environment, run:")
    print("!pip install sqlmodel streamlit pydantic openai google-generativeai python-dateutil pandas plotly python-dotenv pyyaml reportlab anthropic tenacity llama-cpp-python huggingface-hub python-docx")
    print("\nOr ensure you are using the Poetry virtual environment kernel.")

# 1. Setup paths to import from the 'app' package
# Robustly find the smart-job-tracker directory
current_path = Path(os.getcwd())
if current_path.name == "smart-job-tracker":
    smart_job_tracker_dir = current_path
else:
    smart_job_tracker_dir = current_path / "smart-job-tracker"

if str(smart_job_tracker_dir) not in sys.path:
    sys.path.append(str(smart_job_tracker_dir))

# 2. Change working directory to ensure config and models are found correctly
if os.getcwd() != str(smart_job_tracker_dir):
    if smart_job_tracker_dir.exists():
        os.chdir(smart_job_tracker_dir)
    else:
        print(f"❌ Error: Could not find directory at {smart_job_tracker_dir}")

from app.services.extractor import EmailExtractor
from app.models import ApplicationStatus

print(f"Current working directory: {os.getcwd()}")

/home/octavia/.pyenv/versions/3.13.5/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/octavia/GitHub/email-extractor1/smart-job-tracker/app/services/extractor.py:21: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Current working directory: /home/octavia/GitHub/email-extractor1/smart-job-tracker


In [2]:
# 3. Initialize the Extractor
# Note: This will use the providers configured in your config.yaml or .env file.
# If no API keys are provided, it will fallback to local LLM or Heuristics.
extractor = EmailExtractor()
print("Extractor initialized successfully.")

llama_context: n_ctx_per_seq (3072) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


Extractor initialized successfully.


In [3]:
# 4. Define a function to test extraction
def test_extraction(sender, subject, body):
    print(f"\n{'='*50}")
    print(f"SENDER:  {sender}")
    print(f"SUBJECT: {subject}")
    print(f"{'='*50}")
    
    result = extractor.extract(subject, sender, body)
    
    print("\n--- EXTRACTED DATA ---")
    print(json.dumps(result.model_dump(), indent=4, default=str))
    return result

# 5. Run a few experiments
test_emails = [
    {
        "sender": "Google <noreply@google.com>",
        "subject": "Your application for Senior Software Engineer",
        "body": "Hi! Thank you for applying to Google. We have received your application for the Senior Software Engineer role and are currently reviewing it. We will be in touch soon!"
    },
    {
        "sender": "Amazon Recruiting <no-reply@amazon.com>",
        "subject": "Interview Invitation: Software Development Engineer",
        "body": "We are excited to invite you for an interview for the Software Development Engineer position. Please let us know your availability for a 60-minute technical interview next week."
    },
    {
        "sender": "Startup Inc <hiring@startup.io>",
        "subject": "Update on your application",
        "body": "Unfortunately, we have decided not to move forward with your application at this time. We wish you the best of luck in your search."
    }
]

for email in test_emails:
    test_extraction(email['sender'], email['subject'], email['body'])


SENDER:  Google <noreply@google.com>
SUBJECT: Your application for Senior Software Engineer

--- EXTRACTED DATA ---
{
    "company_name": "Google",
    "position": "Senior Software Engineer",
    "status": "APPLIED",
    "summary": "Hi! Thank you for applying to Google.",
    "is_rejection": false,
    "next_step": "We will be in touch soon!"
}

SENDER:  Amazon Recruiting <no-reply@amazon.com>
SUBJECT: Interview Invitation: Software Development Engineer

--- EXTRACTED DATA ---
{
    "company_name": "Amazon",
    "position": "Software Development Engineer",
    "status": "INTERVIEW",
    "summary": "Interview Invitation",
    "is_rejection": false,
    "next_step": "Please let us know your availability for a 60-minute technical interview next week."
}

SENDER:  Startup Inc <hiring@startup.io>
SUBJECT: Update on your application

--- EXTRACTED DATA ---
{
    "company_name": "Startup Inc",
    "position": null,
    "status": "REJECTED",
    "summary": null,
    "is_rejection": true,
    

In [ ]:
# 6. Custom Experiment
# Edit these variables to test your own data!
my_sender = ""
my_subject = ""
my_body = """

"""

if my_sender and my_subject:
    test_extraction(my_sender, my_subject, my_body)
else:
    print("Fill in the variables above to run your own experiment.")